In [ ]:
!pip install unsloth==2025.12.6

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
import torch

In [ ]:
qwen_model, qwen_tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-Coder-3B-Instruct",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)

from peft import PeftModel

qwen_model = PeftModel.from_pretrained(
    qwen_model,
    "Evatos/qwen-sqli-adapters",
)
qwen_tokenizer = get_chat_template(qwen_tokenizer, chat_template="qwen-2.5")

In [ ]:
gemma_model, gemma_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gemma-2-2b-it",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)

from peft import PeftModel

gemma_model = PeftModel.from_pretrained(
    gemma_model,
    "Evatos/gemma-sqli-adapter",
)
gemma_tokenizer = get_chat_template(gemma_tokenizer, chat_template="gemma")

In [ ]:
phi_model, phi_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Phi-3.5-mini-instruct",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)

from peft import PeftModel

phi_model = PeftModel.from_pretrained(
    phi_model,
    "Evatos/phi-sqli-adapter",
)
phi_tokenizer = get_chat_template(phi_tokenizer, chat_template="phi-3")

In [ ]:
FastLanguageModel.for_inference(qwen_model)
FastLanguageModel.for_inference(gemma_model)
FastLanguageModel.for_inference(phi_model)

In [ ]:
SYSTEM_PROMPT = """You are a static code security analyzer.

Your task:
Detect SQL injection vulnerabilities in Python code.
"""

In [ ]:
def analyze(model, tokenizer, code):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": code},
    ]

    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        input_text,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=5,
        temperature=0.0,
        do_sample=False
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    response = response.upper()

    if "DANGEROUS" in response:
        return "DANGEROUS"
    return "SAFE"

In [ ]:
dangerous_code = """
query = "SELECT * FROM users WHERE id='" + user_input + "'"
"""

safe_code = """
def get_user(cursor, user_id):
    return cursor.execute(
        "SELECT id, name FROM users WHERE id = ?",
        (user_id,)
    ).fetchone()
"""

In [ ]:
models = [
    ("qwen", qwen_model, qwen_tokenizer),
    ("gemma", gemma_model, gemma_tokenizer),
    ("phi", phi_model, phi_tokenizer),
]

In [ ]:
def vote(models, code):
    results = []

    for model, tokenizer in models:
        results.append(analyze(model, tokenizer, code))

    cnt = sum(r == "DANGEROUS" for r in results)

    return {
        "results": results,
        "danger_votes": cnt,
        "final": "DANGEROUS" if cnt >= 2 else "SAFE"
    }

In [ ]:
def analyze_fragment(fragment_id, file_name, code):
    results = {}

    for name, model, tokenizer in models:
        result = analyze(model, tokenizer, code)
        results[name] = result

    danger_votes = sum(v == "DANGEROUS" for v in results.values())
    safe_votes = len(models) - danger_votes

    final_label = "DANGEROUS" if danger_votes >= 2 else "SAFE"

    return {
        "fragment_id": fragment_id,
        "file": file_name,
        "code": code,

        "analysis": results,

        "danger_votes": danger_votes,
        "final": final_label,
        "confidence": max(danger_votes, safe_votes) / len(models)
    }

In [ ]:
report = analyze_fragment(
    fragment_id=1,
    file_name="auth.py",
    code=safe_code 
)

print(report)

In [ ]:
pip install fastapi uvicorn

In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class CodeRequest(BaseModel):
    fragment_id: int
    file_name: str
    code: str


@app.post("/analyze")
def analyze_endpoint(req: CodeRequest):
    return analyze_fragment(
        fragment_id=req.fragment_id,
        file_name=req.file_name,
        code=req.code
    )

In [ ]:
!pip install pyngrok

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("3F2CODdlWPj2EhunHwzEMSFjou6_6LVDFw9f4TP3c7sbD8gbW")
public_url = ngrok.connect(8000)
print("PUBLIC URL:", public_url)

In [ ]:
import uvicorn
import asyncio

config = uvicorn.Config(app, host="0.0.0.0", port=8000) 

server = uvicorn.Server(config) 

asyncio.get_event_loop().run_until_complete(server.serve())

In [ ]:
import nest_asyncio
nest_asyncio.apply()

import uvicorn

uvicorn.run(app, host="0.0.0.0", port=8000)

In [ ]:
import threading
import uvicorn

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000)

thread = threading.Thread(target=run_server)
thread.start()

НИЖЕ ОБУЧЕНИЕ МОДЕЛЕЙ

In [ ]:
tokenizer = get_chat_template(tokenizer, chat_template="phi-3")
 
DATASET_PATH = "/kaggle/input/datasets/furquies/sql-injection-ds/sql_injection_ds.jsonl"  # ← поправь путь

PREFIX = "Analyze the following Python code for SQL injection vulnerabilities:\n\n"

raw_dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
print(f"Загружено примеров: {len(raw_dataset)}")
print(f"Поля: {raw_dataset.column_names}")
print(f"Пример:\n{raw_dataset[0]}")
 
 
def convert_to_messages(examples):
    """
    Конвертирует плоские поля instruction/output
    в формат messages + применяет chat template.
    Датасет не имеет system prompt — модель обучается
    отвечать на user-only промпты, что важно для инференса.
    """
    texts = []
    for instruction, output in zip(examples["instruction"], examples["output"]):
        messages = [
            {"role": "user",      "content": PREFIX + instruction},
            {"role": "assistant", "content": output},
        ]
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    return {"text": texts}
 
 
dataset = raw_dataset.map(convert_to_messages, batched=True)
print(f"\nПример после конвертации:\n{dataset[0]['text'][:500]}...")
dataset_split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset_split["train"]
eval_dataset  = dataset_split["test"]
print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset, 
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        #num_train_epochs=3
        max_steps=200,
        warmup_steps=20,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        save_strategy="steps",
        save_steps=10,
        eval_strategy="steps",
        eval_steps=10,
        output_dir="/kaggle/working/sqli_adapter",
        report_to="none",
        max_seq_length=2048,
    ),
)
trainer.train()

In [ ]:
model.save_pretrained("/kaggle/working/sqli_adapter_final")
tokenizer.save_pretrained("/kaggle/working/sqli_adapter_final")